# CUB-200 · Baby Dragon Hatchling · Hebbian explainability bundle

Trains a bird classifier on **CUB-200-2011** with the `hatchvision` framework,
records **Hebbian co-activation** of the BDH sparse-neuron space during training,
clusters the neurons into concepts, **grounds each concept in CUB's 312 visual
attributes** ("wing color: yellow", "bill shape: hooked", ...), and exports the
complete **in-browser inference bundle** for the web app:

- `graph.json` — the Hebbian concept graph (IVGraph)
- `model.onnx` — the classifier with neuron-activation outputs
- `manifest.json` — preprocessing constants + node mapping

**How to run this on Kaggle**
1. *Add Input* → search for the dataset **`wenewone/cub2002011`** (or any mirror of
   the official `CUB_200_2011.tgz`) and attach it. If the notebook has internet
   access it can also download the archive itself (~1.1 GB).
2. *Settings* → **Accelerator: GPU** (T4/P100), **Internet: On** (needed to clone the
   repo and fetch pretrained weights).
3. *Run All* (~30–45 min on a T4 with the default config).
4. Download **`/kaggle/working/bundle.zip`**, unzip its contents into the repo's
   `webapp/` directory, and redeploy — the site then classifies bird photos.

**The universal-tool promise:** nothing below is CUB-specific except the string
`"cub200"` and the attribute display. To rerun this experiment on another dataset,
register a loader for it and change `DATASET` in the config cell — the rest of the
pipeline (training, Hebbian memory, concepts, grounding when attributes exist,
export) is identical.

In [ ]:
# Install hatchvision from the repo
BRANCH = "claude/baby-dragon-classifier-tool-pwtgoj"
!git clone --depth 1 -b {BRANCH} https://github.com/LarsGroep/DragonHatchling.git
%cd DragonHatchling
!pip install -q onnx onnxruntime

import sys, torch
sys.path.insert(0, ".")
import hatchvision
print("hatchvision", hatchvision.__version__, "· torch", torch.__version__,
      "· cuda:", torch.cuda.is_available())

In [ ]:
# ---------------------------- configuration ----------------------------
DATASET      = "cub200"    # any registered loader: swap dataset here, retrain, done
BACKBONE     = "hybrid"    # "hybrid" (pretrained encoder + BDH neurons)  or  "bdh" (pure, from scratch)
EPOCHS       = 12
BATCH_SIZE   = 64
LR           = 1e-3
MAX_UNITS    = 512         # Hebbian tracked neurons (subsampled from neuron_dim)
N_CONCEPTS   = 24
PROBE_IMAGES = 768         # val images used for exemplars + attribute grounding

# per-backbone architecture knobs
BACKBONE_KWARGS = {
    "hybrid": dict(encoder="resnet50", pretrained=True, freeze_encoder=True,
                   neuron_dim=4096),
    # pure BDH from scratch: keep the token count manageable at 128 px
    "bdh":    dict(patch_size=16, dim=256, neuron_dim=2048, depth=6,
                   share_weights=True),
}[BACKBONE]
IMAGE_SIZE = 224 if BACKBONE == "hybrid" else 128
if BACKBONE == "bdh":
    EPOCHS, LR = max(EPOCHS, 30), 3e-4   # from-scratch training needs longer

In [ ]:
# ------------------------- locate / fetch CUB-200 -------------------------
from pathlib import Path

def find_cub_root():
    hits = list(Path("/kaggle/input").glob("**/images.txt"))
    for h in hits:
        if (h.parent / "image_class_labels.txt").exists():
            return h.parent
    return None

root = find_cub_root()
if root is None:
    print("No attached CUB dataset found — downloading (~1.1 GB)…")
    from hatchvision.data.cub import DOWNLOAD_URL
    from torchvision.datasets.utils import download_and_extract_archive
    download_and_extract_archive(DOWNLOAD_URL, "/kaggle/working/cub")
    root = find_cub_root() or Path("/kaggle/working/cub")
print("CUB root:", root)

In [ ]:
from hatchvision import (HebbianFeatureMemory, TrainConfig, Trainer,
                         build_loader, create_model)

data = build_loader(DATASET, root=str(root), image_size=IMAGE_SIZE)
train_loader, val_loader = data.dataloaders(batch_size=BATCH_SIZE, num_workers=2)
print(data.spec.num_classes, "classes ·",
      len(train_loader.dataset), "train /", len(val_loader.dataset), "val")

model = create_model(BACKBONE, data.spec, **BACKBONE_KWARGS)
memory = HebbianFeatureMemory(model, num_classes=data.spec.num_classes,
                              max_units=MAX_UNITS)
print("Hebbian memory observes:", list(model.hebbian_layers()))
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable parameters: {n_train/1e6:.1f} M")

In [ ]:
trainer = Trainer(model, TrainConfig(epochs=EPOCHS, lr=LR, log_every=25), memory)
history = trainer.fit(train_loader, val_loader)
print(f"final val accuracy: {history['val_acc'][-1]:.3f}")

## Concepts: cluster the Hebbian co-activation, ground in attributes

Units that fired together during training are clustered into concepts. Each
concept is then correlated with CUB's per-image attribute annotations over a
probe set, so it gets a name like *"wing color: yellow · bill shape: hooked"*
instead of *"concept 7"*. Datasets without attributes skip this step
automatically and keep class-affinity labels.

In [ ]:
from hatchvision.explain import cluster_concepts, find_exemplars, ground_concepts

layer = memory.layer_names[-1]
concepts = cluster_concepts(memory, layer, data.spec.class_names,
                            n_concepts=N_CONCEPTS)
probe = data.probe_batch(PROBE_IMAGES)
find_exemplars(concepts, memory, model, probe)

attr_names = data.attribute_names()
attr_matrix = data.probe_attributes(probe.shape[0])
if attr_names and attr_matrix is not None:
    ground_concepts(concepts, memory, model, probe, attr_matrix, attr_names)
    print(f"grounded {sum(1 for c in concepts if c.attributes)}/{len(concepts)} concepts")

for c in concepts[:12]:
    attrs = " · ".join(list(c.attributes)[:3]) if c.attributes else "—"
    top_cls = max(c.class_affinity, key=c.class_affinity.get) if c.class_affinity else "—"
    print(f"[{c.concept_id:>2}] {len(c.units):>3} units  coh {c.coherence:.2f}  "
          f"{attrs:<60}  top class: {top_cls}")

In [ ]:
# visual check: exemplar images for the strongest concepts
import matplotlib.pyplot as plt
from hatchvision.explain import denormalize

mean, std = data.spec.normalization()
show = [c for c in concepts if c.exemplars][:6]
fig, axes = plt.subplots(len(show), 6, figsize=(14, 2.4 * len(show)))
for row, c in zip(axes, show):
    for ax, idx in zip(row, c.exemplars):
        img = denormalize(probe[idx:idx + 1], mean, std)[0]
        ax.imshow(img.permute(1, 2, 0))
        ax.axis("off")
    row[0].set_title(c.label, loc="left", fontsize=9)
plt.tight_layout(); plt.show()

## Export the web-app bundle

In [ ]:
from hatchvision.export import export_ivgraph, export_onnx_bundle

OUT = Path("/kaggle/working/bundle")
export_ivgraph(memory, concepts, layer, data.spec.class_names, OUT / "graph.json",
               meta={"dataset": data.spec.name, "backbone": BACKBONE,
                     "val_acc": round(history["val_acc"][-1], 4)})
export_onnx_bundle(model.cpu(), memory, data.spec, OUT,
                   extra_meta={"backbone": BACKBONE})

import shutil
shutil.make_archive("/kaggle/working/bundle", "zip", OUT)
!ls -la /kaggle/working/bundle /kaggle/working/bundle.zip
print("""
Done. Download bundle.zip (right panel -> Output), unzip its three files into
the repo's webapp/ directory (replacing the demo bundle), and redeploy:
    cd webapp && npx vercel deploy --prod
""")

In [ ]:
# optional sanity check: ONNX output matches PyTorch
import onnxruntime, numpy as np, torch
sess = onnxruntime.InferenceSession(str(OUT / "model.onnx"))
x = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE)
model.eval()
with torch.no_grad():
    ref = model(x).numpy()
out = sess.run(None, {"images": x.numpy()})[0]
print("max |onnx - torch| =", float(np.abs(out - ref).max()))